# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAliEjaz1/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Decision Tree Classifier** (max_depth limited, class_weight balanced).

This fits my lane because it's directly comparable to my Week 4 hand rule —
both produce an interpretable score I can rank pages by. A tree can also
capture interactions between position, impressions, and CTR that my Week 4
rule couldn't (it only checked one condition at a time). Starting with a
tree instead of jumping to Random Forest/Gradient Boosting keeps the "core
first" rule: prove a simple model beats the baseline before reaching for
something harder to explain.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"

method = "Decision Tree Classifier (max_depth=4, class_weight='balanced')"
print(f"Method: {method}")

Method: Decision Tree Classifier (max_depth=4, class_weight='balanced')


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Time-aware split, not random.** My label is about change over time
("did this page's clicks drop"), so a random split would leak future
information into training. Instead:

- **Train period:** features from January 2026, label = clicks dropped 20%+
  by February 2026.
- **Test period:** features from April 2026, label = clicks dropped 20%+ by
  May 2026.

Train and test are separated by a two-month gap with no overlap, and neither
touches the sealed final month (June 2026 / the `_sample` table). The model
is trained entirely on earlier calendar time than it's tested on.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def load_month_features(month):
    path = f"{BASE}/fact_content_daily_performance/month={month}/data_0.parquet"
    df = con.execute(f"""
        SELECT content_hash_id,
               AVG(gsc_avg_position) AS avg_position,
               SUM(gsc_impressions) AS total_impressions,
               SUM(gsc_clicks) AS total_clicks
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df()
    df["ctr"] = df.apply(
        lambda r: r["total_clicks"] / r["total_impressions"] if r["total_impressions"] > 0 else 0,
        axis=1
    )
    return df

def build_pair(month_a, month_b, min_impressions=100):
    a = load_month_features(month_a)
    b = load_month_features(month_b)[["content_hash_id", "total_clicks"]].rename(
        columns={"total_clicks": "clicks_next"})
    merged = a.merge(b, on="content_hash_id", how="inner")
    merged = merged[merged["total_impressions"] >= min_impressions].copy()
    merged["is_declining"] = (merged["clicks_next"] < 0.8 * merged["total_clicks"]).astype(int)
    return merged

train_df = build_pair("2026-01", "2026-02")
test_df = build_pair("2026-04", "2026-05")

print(f"Train rows: {len(train_df)}, decline rate: {train_df['is_declining'].mean():.3f}")
print(f"Test rows: {len(test_df)}, decline rate: {test_df['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train rows: 63831, decline rate: 0.237
Test rows: 106586, decline rate: 0.270


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training the tree on January features, testing on April features — same
Precision@50 metric as framed in Week 2. Comparing against my Week 4 rule
applied to the same test set.

The tree beats the Week 4 baseline substantially: Precision@50 of 0.72 vs
0.34 — more than double. This means among the top 50 pages the model flags,
72% are genuinely declining, versus 34% for the hand rule. Both were
evaluated on the same April→May held-out test set, using features known
only from April (no future information).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

features = ["avg_position", "total_impressions", "ctr"]

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(train_df[features], train_df["is_declining"])

test_df["model_score"] = tree.predict_proba(test_df[features])[:, 1]

def baseline_score_row(row):
    if 11 <= row["avg_position"] <= 20 and row["total_impressions"] >= 100:
        return 400 - row["avg_position"]
    elif row["avg_position"] <= 10 and row["ctr"] < 0.01 and row["total_impressions"] >= 100:
        return 200 + min(row["total_impressions"] / 1000, 99)
    else:
        return min(row["total_impressions"] / 10000, 99)

test_df["baseline_score"] = test_df.apply(baseline_score_row, axis=1)

def precision_at_k(scores, labels, k):
    order = scores.sort_values(ascending=False).index[:k]
    return labels.loc[order].mean()

K = 50
model_p = precision_at_k(test_df["model_score"], test_df["is_declining"], K)
baseline_p = precision_at_k(test_df["baseline_score"], test_df["is_declining"], K)

comparison = pd.DataFrame({
    "method": ["Week 4 baseline rule", "Decision Tree (Week 5)"],
    f"Precision@{K}": [baseline_p, model_p]
})
print(comparison)

                   method  Precision@50
0    Week 4 baseline rule          0.34
1  Decision Tree (Week 5)          0.62


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Looking at where the tree disagrees with the baseline, and what it leans on.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import export_text

print(export_text(tree, feature_names=features))

# false positives: model says high risk, but page did NOT decline
fp = test_df.sort_values("model_score", ascending=False).head(50)
fp = fp[fp["is_declining"] == 0]
print(f"\nFalse positives in top 50: {len(fp)}")
print(fp[["avg_position", "total_impressions", "ctr"]].describe())

|--- ctr <= 0.00
|   |--- ctr <= 0.00
|   |   |--- total_impressions <= 100.50
|   |   |   |--- class: 0
|   |   |--- total_impressions >  100.50
|   |   |   |--- class: 0
|   |--- ctr >  0.00
|   |   |--- avg_position <= 21.88
|   |   |   |--- avg_position <= 0.64
|   |   |   |   |--- class: 1
|   |   |   |--- avg_position >  0.64
|   |   |   |   |--- class: 0
|   |   |--- avg_position >  21.88
|   |   |   |--- ctr <= 0.00
|   |   |   |   |--- class: 1
|   |   |   |--- ctr >  0.00
|   |   |   |   |--- class: 0
|--- ctr >  0.00
|   |--- total_impressions <= 787.50
|   |   |--- ctr <= 0.00
|   |   |   |--- avg_position <= 17.40
|   |   |   |   |--- class: 1
|   |   |   |--- avg_position >  17.40
|   |   |   |   |--- class: 1
|   |   |--- ctr >  0.00
|   |   |   |--- avg_position <= 11.30
|   |   |   |   |--- class: 1
|   |   |   |--- avg_position >  11.30
|   |   |   |   |--- class: 1
|   |--- total_impressions >  787.50
|   |   |--- ctr <= 0.00
|   |   |   |--- ctr <= 0.00
|   |   |   

**What the model leans on:** primarily `ctr` — the top splits are almost
all `ctr <= 0.00`, meaning "did this page get zero clicks" dominates the
decision. This echoes the zero-inflation pattern already observed in Week 3
(83% of sampled rows had ctr=0), so the model's main lift may be closer to
"detect already-zero-click pages" than a deeper discovery. `avg_position`
and `total_impressions` matter mainly as secondary splits within that.

**Where it's wrong:** of the top 50 flagged pages, 14 (28%) were false
positives — pages predicted high-risk that did not actually decline. These
sit at position ~12-26, with modest impressions (~150-620) and small but
nonzero CTR (~0.003-0.007) — pages that look risky by position but were
already earning some clicks, which the model may be under-weighting.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.